# 🔧 Notebook 03 — Feature Engineering

**Purpose:** Transform raw cleaned columns into model-ready numeric features. Every decision is explained and justified. The feature matrix is saved as parquet for notebook 04.

**Key principle:** All encoders (target encoder, OHE) are *fit on training data only* and applied to both train and test. This prevents target leakage through the feature engineering pipeline.

## Setup

In [16]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

OUTPUTS_PATH = "../outputs"
FIGURES_PATH = os.path.join(OUTPUTS_PATH, "figures")
os.makedirs(FIGURES_PATH, exist_ok=True)

# Load cleaned splits from Notebook 01
train_df = pd.read_parquet(os.path.join(OUTPUTS_PATH, "train_df.parquet"))
test_df  = pd.read_parquet(os.path.join(OUTPUTS_PATH, "test_df.parquet"))

print(f"Train : {train_df.shape}")
print(f"Test  : {test_df.shape}")
print(f"\nColumns available: {train_df.columns.tolist()}")


Train : (150743, 20)
Test  : (37686, 20)

Columns available: ['blurb', 'country', 'country_displayable_name', 'created_at', 'currency', 'deadline', 'disable_communication', 'goal', 'id', 'is_launched', 'launched_at', 'name', 'staff_pick', 'state', 'video', 'success', 'cat_name', 'cat_parent_name', 'loc_country', 'loc_state']


## Feature 1 — `log_goal`

`goal` has extreme right skew — the maximum is in the hundreds of millions. A log transform compresses this to a manageable range, brings the distribution closer to normal, and prevents extreme outliers from dominating tree splits or linear model coefficients.

`log_goal = log(1 + goal)` — the +1 handles the edge case of goal = 0.

In [17]:
for df in [train_df, test_df]:
    goal = pd.to_numeric(df["goal"], errors="coerce").fillna(0).clip(lower=0)
    df["log_goal"] = np.log1p(goal)

print("log_goal stats (train):")
print(train_df["log_goal"].describe())
print(f"\nOriginal goal range: 0 to {pd.to_numeric(train_df['goal'], errors='coerce').max():,.0f}")
print(f"log_goal range    : {train_df['log_goal'].min():.2f} to {train_df['log_goal'].max():.2f}")


log_goal stats (train):
count    150743.000000
mean          8.439301
std           1.782361
min           0.009950
25%           7.313887
50%           8.517393
75%           9.615872
max          20.723266
Name: log_goal, dtype: float64

Original goal range: 0 to 1,000,000,000
log_goal range    : 0.01 to 20.72


## Feature 2 — `duration_days`

Campaign length = (deadline - launched_at) / 86,400 seconds per day. Kickstarter's maximum allowed campaign duration is 90 days, so we cap there. Shorter campaigns historically succeed more — this feature is one of the strongest predictors in the dataset.

In [18]:
for df in [train_df, test_df]:
    launched = pd.to_numeric(df["launched_at"], errors="coerce")
    deadline = pd.to_numeric(df["deadline"], errors="coerce")
    df["duration_days"] = ((deadline - launched) / 86400).clip(0, 90).astype(float)

print("duration_days stats (train):")
print(train_df["duration_days"].describe())
print(f"\nCampaigns exactly at 30 days: {(train_df['duration_days'].round() == 30).sum():,}")
print(f"Campaigns exactly at 60 days: {(train_df['duration_days'].round() == 60).sum():,}")


duration_days stats (train):
count    150743.000000
mean         33.550978
std          12.673708
min           1.000000
25%          29.958333
50%          30.000000
75%          36.315394
max          90.000000
Name: duration_days, dtype: float64

Campaigns exactly at 30 days: 62,619
Campaigns exactly at 60 days: 16,395


## Feature 3 — `prep_days`

Days between account creation and campaign launch. A creator who spends 3 months preparing their campaign likely puts in more effort than one who launches the same day they signed up. We winsorise at the 99th percentile because some creators created accounts years before Kickstarter even had their category — these are long-tail outliers unrelated to campaign effort.

In [19]:
for df in [train_df, test_df]:
    created  = pd.to_numeric(df.get("created_at", pd.Series(np.nan, index=df.index)), errors="coerce")
    launched = pd.to_numeric(df["launched_at"], errors="coerce")
    raw = ((launched - created) / 86400).clip(lower=0)
    cap = train_df.apply(lambda r: ((pd.to_numeric(r.get("launched_at",0), errors="coerce") - 
                                      pd.to_numeric(r.get("created_at",0), errors="coerce")) / 86400), 
                          axis=1).clip(lower=0).quantile(0.99)
    df["prep_days"] = raw.clip(upper=cap).astype(float)

print("prep_days stats (train, after winsorisation):")
print(train_df["prep_days"].describe())


prep_days stats (train, after winsorisation):
count    150743.000000
mean         44.594412
std          97.958847
min           0.002731
25%           3.447315
50%          12.593600
75%          37.974734
max         670.665410
Name: prep_days, dtype: float64


## Feature 4 — `has_video` (binary)

A missing `video` column means the creator did not attach a video — this is a deliberate choice, not missing data. We encode it as a binary feature: 1 = video present, 0 = no video. From the EDA, campaigns with videos succeed at a significantly higher rate.

In [20]:
for df in [train_df, test_df]:
    if "video" in df.columns:
        df["has_video"] = df["video"].notna().astype(int)
    else:
        df["has_video"] = 0

print("has_video distribution (train):")
print(train_df["has_video"].value_counts())
print(f"\nSuccess rate with video   : {train_df[train_df['has_video']==1]['success'].mean()*100:.1f}%")
print(f"Success rate without video: {train_df[train_df['has_video']==0]['success'].mean()*100:.1f}%")


has_video distribution (train):
has_video
1    102098
0     48645
Name: count, dtype: int64

Success rate with video   : 68.5%
Success rate without video: 44.5%


## Features 5–8 — Text Features

The `blurb` (short description) and `name` contain signal about creator effort and communication quality. We extract simple length-based features — a proper NLP approach would use TF-IDF or embeddings, but length alone captures a meaningful portion of the variance.

- **blurb_length**: character count — a longer blurb usually means more explanation
- **name_length**: character count of the title
- **blurb_word_count**: word count (different from char count — captures avg word length)
- **name_has_number**: campaigns like 'Volume 2' or 'Season 3' signal an established creator with a prior audience

In [21]:
for df in [train_df, test_df]:
    df["blurb_length"]     = df["blurb"].fillna("").str.len().astype(int)
    df["name_length"]      = df["name"].fillna("").str.len().astype(int)
    df["blurb_word_count"] = df["blurb"].fillna("").str.split().str.len().fillna(0).astype(int)
    df["name_has_number"]  = df["name"].fillna("").str.contains(r"\d", regex=True).astype(int)

print("Text feature stats (train):")
print(train_df[["blurb_length","name_length","blurb_word_count","name_has_number"]].describe().T)
print(f"\nSuccess rate — name has number    : {train_df[train_df['name_has_number']==1]['success'].mean()*100:.1f}%")
print(f"Success rate — name no number     : {train_df[train_df['name_has_number']==0]['success'].mean()*100:.1f}%")


Text feature stats (train):
                     count        mean        std  min   25%    50%    75%  \
blurb_length      150743.0  105.839555  31.086269  0.0  87.0  119.0  131.0   
name_length       150743.0   34.619624  15.499379  1.0  22.0   34.0   48.0   
blurb_word_count  150743.0   17.641867   5.654482  0.0  14.0   19.0   22.0   
name_has_number   150743.0    0.132888   0.339455  0.0   0.0    0.0    0.0   

                    max  
blurb_length      151.0  
name_length        85.0  
blurb_word_count   43.0  
name_has_number     1.0  

Success rate — name has number    : 71.8%
Success rate — name no number     : 59.0%


## Features 9–13 — Campaign Metadata Features

- **goal_is_round**: Round goals ($10,000) may signal less precise budgeting. A creator who asks for exactly $8,750 has probably costed their project carefully.
- **is_usd**: USD campaigns dominate the platform and may have different dynamics (larger audience, no currency friction).
- **launched_month** (1–12): Seasonal effects — Q4 pre-holiday launches may attract more backers.
- **launched_dayofweek** (0=Mon, 6=Sun): Tuesday/Wednesday launches have historically performed best.
- **launched_year**: Captures platform maturity and economic conditions at time of launch.

In [22]:
for df in [train_df, test_df]:
    goal = pd.to_numeric(df["goal"], errors="coerce").fillna(0)
    df["goal_is_round"] = ((goal % 1000) == 0).astype(int)
    
    if "currency" in df.columns:
        df["is_usd"] = (df["currency"] == "USD").astype(int)
    else:
        df["is_usd"] = 0
    
    launched_dt = pd.to_datetime(pd.to_numeric(df["launched_at"], errors="coerce"), unit="s", utc=True)
    df["launched_month"]     = launched_dt.dt.month.astype(float)
    df["launched_dayofweek"] = launched_dt.dt.dayofweek.astype(float)
    df["launched_year"]      = launched_dt.dt.year.astype(float)

print("New metadata features (train):")
print(train_df[["goal_is_round","is_usd","launched_month","launched_dayofweek","launched_year"]].describe().T)
print(f"\nSuccess rate — round goal: {train_df[train_df['goal_is_round']==1]['success'].mean()*100:.1f}%")
print(f"Success rate — non-round : {train_df[train_df['goal_is_round']==0]['success'].mean()*100:.1f}%")


New metadata features (train):
                       count         mean       std     min     25%     50%  \
goal_is_round       150743.0     0.585288  0.492674     0.0     0.0     1.0   
is_usd              150743.0     0.671739  0.469582     0.0     0.0     1.0   
launched_month      150743.0     6.240761  3.343954     1.0     3.0     6.0   
launched_dayofweek  150743.0     2.424902  1.808133     0.0     1.0     2.0   
launched_year       150743.0  2017.802983  3.543219  2009.0  2015.0  2017.0   

                       75%     max  
goal_is_round          1.0     1.0  
is_usd                 1.0     1.0  
launched_month         9.0    12.0  
launched_dayofweek     4.0     6.0  
launched_year       2021.0  2024.0  

Success rate — round goal: 55.4%
Success rate — non-round : 68.2%


## Feature 14 — `staff_pick` (binary int)

Staff pick is already boolean — we just ensure it's encoded as 0/1 integer for sklearn compatibility.

In [23]:
for df in [train_df, test_df]:
    if "staff_pick" in df.columns:
        sp = df["staff_pick"]
        df["staff_pick"] = sp.map({"True": 1, "False": 0, True: 1, False: 0}).fillna(0).astype(int)

print("staff_pick distribution (train):")
print(train_df["staff_pick"].value_counts())


staff_pick distribution (train):
staff_pick
0    127475
1     23268
Name: count, dtype: int64


## Feature 15 — `goal_per_day`

How much money must be raised per day to hit the goal. A campaign asking for $50,000 over 5 days needs $10,000/day — a much harder ask than the same goal over 60 days ($833/day). This combines two strong individual features (goal and duration) into a joint intensity measure.

In [24]:
for df in [train_df, test_df]:
    goal = pd.to_numeric(df["goal"], errors="coerce").fillna(0)
    dur  = df["duration_days"].replace(0, np.nan)
    df["goal_per_day"] = (goal / dur).fillna(0).clip(upper=goal.quantile(0.99)).astype(float)

print("goal_per_day stats (train):")
print(train_df["goal_per_day"].describe())


goal_per_day stats (train):
count    150743.000000
mean       1102.328121
std       11749.532185
min           0.001115
25%          50.000000
50%         163.090197
75%         428.571429
max      444257.520000
Name: goal_per_day, dtype: float64


## Feature 16 — `is_california`

California (SF Bay Area + LA) is the largest creator hub on Kickstarter. These campaigns may benefit from stronger tech press coverage, VC networks, and a denser creator community.

In [25]:
for df in [train_df, test_df]:
    if "loc_state" in df.columns:
        df["is_california"] = (df["loc_state"] == "CA").astype(int)
    else:
        df["is_california"] = 0

print(f"is_california distribution (train):")
print(train_df["is_california"].value_counts())
print(f"\nCA success rate : {train_df[train_df['is_california']==1]['success'].mean()*100:.1f}%")
print(f"Non-CA rate     : {train_df[train_df['is_california']==0]['success'].mean()*100:.1f}%")


is_california distribution (train):
is_california
0    132963
1     17780
Name: count, dtype: int64

CA success rate : 67.1%
Non-CA rate     : 59.9%


## Feature 17 — Target Encoding for `cat_name`

With 160+ subcategories, one-hot encoding would create 160 sparse binary columns — mostly zeros — that would slow training and potentially overfit to rare categories.

**Target encoding** replaces each category with its mean success rate. We use *smoothed target encoding*:

```
encoded = (n × cat_mean + λ × global_mean) / (n + λ)
```

where λ (smoothing) controls how much we shrink rare categories toward the global mean.

**⚠️ Critical:** The encoder is fit on **training data only**. If we encoded using the full dataset, the test set's success rate would leak into its own features. We never fit on the test set.

In [26]:
def fit_target_encoder(train_df, col, target="success", smoothing=10.0):
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(["mean", "count"])
    encoder = {}
    for cat, row in stats.iterrows():
        n = row["count"]
        encoded = (n * row["mean"] + smoothing * global_mean) / (n + smoothing)
        encoder[cat] = encoded
    encoder["__global_mean__"] = global_mean
    return encoder

def apply_target_encoder(df, col, encoder, new_col):
    global_mean = encoder.get("__global_mean__", 0.5)
    df[new_col] = df[col].map(encoder).fillna(global_mean)
    return df

# Fit on training data ONLY
cat_name_encoder = fit_target_encoder(train_df, "cat_name")

# Apply to both (test uses training statistics)
train_df = apply_target_encoder(train_df, "cat_name", cat_name_encoder, "cat_name_encoded")
test_df  = apply_target_encoder(test_df,  "cat_name", cat_name_encoder, "cat_name_encoded")

print("cat_name_encoded (target-encoded) stats (train):")
print(train_df["cat_name_encoded"].describe())
print(f"\nTop 10 subcategories by encoded value (success rate):")
top = sorted([(k,v) for k,v in cat_name_encoder.items() if k != "__global_mean__"], key=lambda x: -x[1])[:10]
for k,v in top:
    print(f"  {k:<35} : {v*100:.1f}%")


cat_name_encoded (target-encoded) stats (train):
count    150743.000000
mean          0.607770
std           0.266390
min           0.160163
25%           0.390873
50%           0.559109
75%           0.905340
max           0.995874
Name: cat_name_encoded, dtype: float64

Top 10 subcategories by encoded value (success rate):
  Dance                               : 99.6%
  Indie Rock                          : 99.4%
  Country & Folk                      : 99.3%
  Hardware                            : 98.9%
  Crafts                              : 98.9%
  Narrative Film                      : 98.9%
  Theater                             : 98.8%
  Rock                                : 98.7%
  Pop                                 : 98.6%
  Art Books                           : 98.5%


## Feature 18 — One-Hot Encode `cat_parent_name`

Only 16 parent categories — manageable with OHE. Fit on training data, apply to both.

In [27]:
# Fit on training
parent_categories = sorted(train_df["cat_parent_name"].dropna().unique().tolist())
print(f"Parent categories ({len(parent_categories)}): {parent_categories}")

for df in [train_df, test_df]:
    df["cat_parent_name"] = df["cat_parent_name"].fillna("Unknown")
    for cat in parent_categories:
        col = "parent_" + cat.replace(" ", "_").replace("/", "_")
        df[col] = (df["cat_parent_name"] == cat).astype(int)

parent_cols = ["parent_" + c.replace(" ", "_").replace("/", "_") for c in parent_categories]
print(f"\nAdded {len(parent_cols)} OHE parent category columns")
print(train_df[parent_cols].sum().sort_values(ascending=False).head(10))


Parent categories (15): ['Art', 'Comics', 'Crafts', 'Dance', 'Design', 'Fashion', 'Film & Video', 'Food', 'Games', 'Journalism', 'Music', 'Photography', 'Publishing', 'Technology', 'Theater']

Added 15 OHE parent category columns
parent_Music           23675
parent_Film_&_Video    22459
parent_Publishing      15776
parent_Food            13393
parent_Art             13213
parent_Technology      12706
parent_Fashion          8664
parent_Crafts           6286
parent_Photography      5771
parent_Games            5676
dtype: int64


## Feature 19 — One-Hot Encode `country`

Group countries with fewer than 500 campaigns into 'Other' to avoid sparse, unreliable columns. Fit on training data only.

In [28]:
# Group rare countries
country_counts = train_df["country"].value_counts()
keep_countries = country_counts[country_counts >= 500].index.tolist()
print(f"Countries with ≥500 campaigns: {keep_countries}")
print(f"Countries grouped into 'Other': {country_counts[country_counts < 500].index.tolist()}")

for df in [train_df, test_df]:
    df["country_grouped"] = df["country"].where(df["country"].isin(keep_countries), other="Other")
    country_dummies = pd.get_dummies(df["country_grouped"], prefix="country").astype(int)
    for col in country_dummies.columns:
        df[col] = country_dummies[col]

country_cols = [c for c in train_df.columns if c.startswith("country_") and train_df[c].dtype != object]
print(f"\nAdded {len(country_cols)} country OHE columns: {country_cols}")


Countries with ≥500 campaigns: ['US', 'GB', 'CA', 'AU', 'DE', 'MX', 'FR', 'IT', 'ES', 'HK', 'NL', 'SE', 'JP', 'DK', 'NZ', 'SG', 'BE', 'CH', 'IE']
Countries grouped into 'Other': ['AT', 'NO', 'PL', 'GR', 'LU', 'SI']

Added 20 country OHE columns: ['country_AU', 'country_BE', 'country_CA', 'country_CH', 'country_DE', 'country_DK', 'country_ES', 'country_FR', 'country_GB', 'country_HK', 'country_IE', 'country_IT', 'country_JP', 'country_MX', 'country_NL', 'country_NZ', 'country_Other', 'country_SE', 'country_SG', 'country_US']


## Feature Summary Table

In [29]:
engineered_features = [
    ("log_goal",          "numeric",     "Log(1+goal) — compresses right skew",               False),
    ("duration_days",     "numeric",     "Campaign duration in days (capped at 90)",           True),
    ("prep_days",         "numeric",     "Days from account creation to launch (winsorised)",  True),
    ("has_video",         "binary",      "1 = video attached, 0 = no video",                  True),
    ("blurb_length",      "numeric",     "Character count of short description",               True),
    ("name_length",       "numeric",     "Character count of campaign title",                  True),
    ("blurb_word_count",  "numeric",     "Word count of short description",                    True),
    ("name_has_number",   "binary",      "1 = title contains a digit",                        True),
    ("goal_is_round",     "binary",      "1 = goal is divisible by 1000",                     True),
    ("is_usd",            "binary",      "1 = currency is USD",                               True),
    ("launched_month",    "numeric",     "Month of launch (1–12)",                            True),
    ("launched_dayofweek","numeric",     "Day of week (0=Mon, 6=Sun)",                        True),
    ("launched_year",     "numeric",     "Year of launch",                                    True),
    ("staff_pick",        "binary",      "1 = editorially featured by Kickstarter",           False),
    ("goal_per_day",      "numeric",     "goal / duration_days (daily fundraising target)",   True),
    ("is_california",     "binary",      "1 = campaign located in California",                True),
    ("cat_name_encoded",  "numeric",     "Target-encoded subcategory (train mean success rate)", True),
]

summary = pd.DataFrame(engineered_features,
    columns=["feature", "type", "description", "engineered"])
summary["missing_rate_train"] = summary["feature"].apply(
    lambda f: f"{train_df[f].isnull().mean()*100:.2f}%" if f in train_df.columns else "N/A"
)
print(summary.to_string(index=False))
os.makedirs(os.path.join(OUTPUTS_PATH, "results"), exist_ok=True)
summary.to_csv(os.path.join(OUTPUTS_PATH, "results", "feature_summary.csv"), index=False)
print(f"\nSaved feature_summary.csv")


           feature    type                                          description  engineered missing_rate_train
          log_goal numeric                  Log(1+goal) — compresses right skew       False              0.00%
     duration_days numeric             Campaign duration in days (capped at 90)        True              0.00%
         prep_days numeric    Days from account creation to launch (winsorised)        True              0.00%
         has_video  binary                     1 = video attached, 0 = no video        True              0.00%
      blurb_length numeric                 Character count of short description        True              0.00%
       name_length numeric                    Character count of campaign title        True              0.00%
  blurb_word_count numeric                      Word count of short description        True              0.00%
   name_has_number  binary                           1 = title contains a digit        True              0.00%
 

## Build Final Feature Matrix and Save

In [30]:
BASE_FEATURES = [
    "log_goal", "duration_days", "prep_days", "has_video",
    "blurb_length", "name_length", "blurb_word_count", "name_has_number",
    "goal_is_round", "is_usd", "launched_month", "launched_dayofweek",
    "launched_year", "staff_pick", "goal_per_day", "is_california",
    "cat_name_encoded",
]
OHE_FEATURES = (
    ["parent_" + c.replace(" ", "_").replace("/", "_") for c in parent_categories] +
    [c for c in train_df.columns if c.startswith("country_") and train_df[c].dtype != object]
)

FEATURE_COLS = [f for f in BASE_FEATURES + OHE_FEATURES if f in train_df.columns]
print(f"Total features: {len(FEATURE_COLS)}")
print(f"Feature list: {FEATURE_COLS}")

X_train = train_df[FEATURE_COLS].fillna(0)
y_train = train_df["success"]
X_test  = test_df[FEATURE_COLS].fillna(0)
y_test  = test_df["success"]

print(f"\nX_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   y_test : {y_test.shape}")

# Save feature column list for Notebook 04
import json
with open(os.path.join(OUTPUTS_PATH, "feature_cols.json"), "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)

X_train.to_parquet(os.path.join(OUTPUTS_PATH, "X_train.parquet"), index=False)
X_test.to_parquet(os.path.join(OUTPUTS_PATH,  "X_test.parquet"),  index=False)
y_train.to_frame().to_parquet(os.path.join(OUTPUTS_PATH, "y_train.parquet"), index=False)
y_test.to_frame().to_parquet(os.path.join(OUTPUTS_PATH,  "y_test.parquet"),  index=False)

print("\nSaved: X_train.parquet, X_test.parquet, y_train.parquet, y_test.parquet")
print("Notebook 03 complete. Proceed to 04_modelling.ipynb")


Total features: 52
Feature list: ['log_goal', 'duration_days', 'prep_days', 'has_video', 'blurb_length', 'name_length', 'blurb_word_count', 'name_has_number', 'goal_is_round', 'is_usd', 'launched_month', 'launched_dayofweek', 'launched_year', 'staff_pick', 'goal_per_day', 'is_california', 'cat_name_encoded', 'parent_Art', 'parent_Comics', 'parent_Crafts', 'parent_Dance', 'parent_Design', 'parent_Fashion', 'parent_Film_&_Video', 'parent_Food', 'parent_Games', 'parent_Journalism', 'parent_Music', 'parent_Photography', 'parent_Publishing', 'parent_Technology', 'parent_Theater', 'country_AU', 'country_BE', 'country_CA', 'country_CH', 'country_DE', 'country_DK', 'country_ES', 'country_FR', 'country_GB', 'country_HK', 'country_IE', 'country_IT', 'country_JP', 'country_MX', 'country_NL', 'country_NZ', 'country_Other', 'country_SE', 'country_SG', 'country_US']

X_train: (150743, 52)  y_train: (150743,)
X_test : (37686, 52)   y_test : (37686,)

Saved: X_train.parquet, X_test.parquet, y_train.pa